In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
import qnt.data as qndata


# ============================================================
# 1) Load Quantiacs data + choose random liquid assets
# ============================================================
def load_quantiacs_universe(seed=7, n_assets=50, data=None):
    if data is None:
        data = qndata.stocks.load_data()

    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    liquid_last = is_liquid.isel(time=-1).fillna(0).astype(bool)
    liquid_assets = close.asset.values[liquid_last.values]

    if len(liquid_assets) < n_assets:
        raise ValueError(f"Not enough liquid assets ({len(liquid_assets)}) to pick {n_assets}.")

    rng = np.random.default_rng(seed)
    chosen_assets = rng.choice(liquid_assets, size=n_assets, replace=False)

    return data, chosen_assets


# ============================================================
# 2) Returns, rolling mean and covariance
# ============================================================
def compute_returns_from_close(close, use_log_returns=False):
    if use_log_returns:
        rets = np.log(close / close.shift(time=1))
    else:
        rets = close / close.shift(time=1) - 1

    rets = rets.dropna("time")
    times = rets.time.values
    R = rets.values.astype(float)
    return times, R


def compute_mu_sigma_series(data, chosen_assets, lookback=252, annualise=252, use_log_returns=False):
    close = data.sel(field="close").sel(asset=chosen_assets)
    times, R = compute_returns_from_close(close, use_log_returns=use_log_returns)

    T, n = R.shape
    if T <= lookback + 1:
        raise ValueError(f"Not enough history: T={T}, need > {lookback + 1}.")

    mu_series = np.zeros((T, n))
    Sigma_series = np.zeros((T, n, n))

    for t in range(T):
        start = max(0, t - lookback + 1)
        window = R[start:t + 1, :]

        mu = np.nanmean(window, axis=0)

        if window.shape[0] >= 2:
            Sigma = np.cov(window, rowvar=False)
            if np.ndim(Sigma) == 0:
                Sigma = np.array([[Sigma]])
        else:
            Sigma = np.eye(n) * 1e-6

        mu_series[t] = mu * annualise
        Sigma_series[t] = Sigma * annualise

    return times, mu_series, Sigma_series, R


# ============================================================
# 3) Helpers
# ============================================================
def make_psd_matrix(Sigma, ridge=1e-3, eps=1e-8):
    S = 0.5 * (Sigma + Sigma.T)
    S = S + ridge * np.eye(S.shape[0]) + eps * np.eye(S.shape[0])
    return S


def add_constraints(w, long_only=True, w_min=None, w_max=None, fully_invested=True):
    cons = []

    if fully_invested:
        cons.append(cp.sum(w) == 1)

    if long_only:
        cons.append(w >= 0)

    if w_min is not None:
        cons.append(w >= w_min)

    if w_max is not None:
        cons.append(w <= w_max)

    return cons


def solve_problem(prob):
    for solver in [cp.OSQP, cp.SCS, cp.ECOS]:
        try:
            prob.solve(solver=solver, warm_start=True, verbose=False)
            if prob.status in ["optimal", "optimal_inaccurate"]:
                return True
        except Exception:
            pass
    return False


def normalise_weights(w, fallback=None):
    if w is None:
        if fallback is None:
            raise ValueError("No solution and no fallback provided.")
        w = fallback.copy()

    w = np.asarray(w, dtype=float)
    w = np.nan_to_num(w, nan=0.0, posinf=0.0, neginf=0.0)
    w[w < 0] = 0.0

    s = w.sum()
    if s <= 0:
        if fallback is None:
            w = np.ones_like(w) / len(w)
        else:
            w = np.maximum(fallback, 0.0)
            s2 = w.sum()
            if s2 <= 0:
                w = np.ones_like(w) / len(w)
            else:
                w = w / s2
    else:
        w = w / s

    return w


def turnover_from_weights(W):
    if W.shape[0] < 2:
        return np.array([])
    return np.sum(np.abs(np.diff(W, axis=0)), axis=1)


def equity_curve(rp, start=1.0):
    return start * np.cumprod(1.0 + rp)


def max_drawdown(eq):
    peak = np.maximum.accumulate(eq)
    dd = eq / peak - 1.0
    return np.min(dd)


def summary_stats(rp, periods_per_year=252):
    rp = np.asarray(rp, dtype=float)
    rp = rp[np.isfinite(rp)]

    if len(rp) == 0:
        return {
            "n_obs": 0,
            "mean_daily": np.nan,
            "vol_daily": np.nan,
            "ann_return": np.nan,
            "ann_vol": np.nan,
            "sharpe": np.nan,
            "max_drawdown": np.nan,
            "cum_return": np.nan,
        }

    mean_daily = np.mean(rp)
    vol_daily = np.std(rp, ddof=1) if len(rp) > 1 else 0.0
    ann_return = mean_daily * periods_per_year
    ann_vol = vol_daily * np.sqrt(periods_per_year)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    eq = equity_curve(rp)
    mdd = max_drawdown(eq)
    cum_return = eq[-1] - 1.0

    return {
        "n_obs": len(rp),
        "mean_daily": mean_daily,
        "vol_daily": vol_daily,
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": mdd,
        "cum_return": cum_return,
    }


def print_stats(name, stats):
    print(f"\n{name}")
    print("-" * len(name))
    for k, v in stats.items():
        if isinstance(v, (float, np.floating)):
            print(f"{k:15s}: {v: .6f}")
        else:
            print(f"{k:15s}: {v}")


# ============================================================
# 4) Time-varying transaction cost series
# ============================================================
def build_time_varying_cost_series(
    R,
    base_cost=0.0020,
    mode="vol_scaled",
    vol_lookback=21,
    alpha=2.0
):
    T, _ = R.shape

    if mode == "constant":
        return np.full(T, base_cost)

    if mode == "vol_scaled":
        market_abs = np.mean(np.abs(R), axis=1)

        vol_state = np.zeros(T)
        for t in range(T):
            start = max(0, t - vol_lookback + 1)
            vol_state[t] = np.mean(market_abs[start:t + 1])

        denom = np.nanmean(vol_state)
        if denom <= 0:
            denom = 1.0

        vol_state_norm = vol_state / denom
        cost_series = base_cost * (1.0 + alpha * (vol_state_norm - 1.0))
        cost_series = np.maximum(cost_series, 0.0001)
        return cost_series

    raise ValueError("mode must be 'constant' or 'vol_scaled'")


# ============================================================
# 5) BASIC one-step
#    max mu'w - gamma w'Sw - lam_t ||w-w_prev||_1
# ============================================================
def solve_basic_one_step(
    mu,
    Sigma,
    w_prev,
    gamma=5.0,
    lam_t=0.0020,
    w_max=0.10,
    ridge=1e-3
):
    n = len(mu)
    w = cp.Variable(n)

    S = make_psd_matrix(Sigma, ridge=ridge)
    trade = w - w_prev

    objective = cp.Maximize(
        mu @ w
        - gamma * cp.quad_form(w, S)
        - lam_t * cp.norm1(trade)
    )

    constraints = add_constraints(
        w,
        long_only=True,
        w_max=w_max,
        fully_invested=True
    )

    prob = cp.Problem(objective, constraints)
    ok = solve_problem(prob)

    if not ok:
        return normalise_weights(w_prev)

    return normalise_weights(w.value, fallback=w_prev)


# ============================================================
# 6) BASIC two-step
#    Step 1: target w*
#    Step 2: min (rho/2)||w-w*||^2 + lam_t ||w-w_prev||_1
# ============================================================
def solve_basic_two_step(
    mu,
    Sigma,
    w_prev,
    gamma=5.0,
    rho=25.0,
    lam_t=0.0020,
    w_max=0.10,
    ridge=1e-3
):
    n = len(mu)
    S = make_psd_matrix(Sigma, ridge=ridge)

    # Step 1: target portfolio
    w_star = cp.Variable(n)
    objective1 = cp.Maximize(
        mu @ w_star - gamma * cp.quad_form(w_star, S)
    )
    constraints1 = add_constraints(
        w_star,
        long_only=True,
        w_max=w_max,
        fully_invested=True
    )

    prob1 = cp.Problem(objective1, constraints1)
    ok1 = solve_problem(prob1)

    if not ok1:
        w_star_val = normalise_weights(w_prev)
    else:
        w_star_val = normalise_weights(w_star.value, fallback=w_prev)

    # Step 2: controlled move towards target
    w = cp.Variable(n)
    trade = w - w_prev

    objective2 = cp.Minimize(
        (rho / 2.0) * cp.sum_squares(w - w_star_val)
        + lam_t * cp.norm1(trade)
    )
    constraints2 = add_constraints(
        w,
        long_only=True,
        w_max=w_max,
        fully_invested=True
    )

    prob2 = cp.Problem(objective2, constraints2)
    ok2 = solve_problem(prob2)

    if not ok2:
        w_val = normalise_weights(w_prev)
    else:
        w_val = normalise_weights(w.value, fallback=w_prev)

    return w_star_val, w_val


# ============================================================
# 7) ENHANCED one-step
#    max mu'w - gamma w'Sw - lam_t ||w-w_prev||_1 - eta ||w-w_prev||_2^2
#    s.t. ||w-w_prev||_1 <= tau
# ============================================================
def solve_enhanced_one_step(
    mu,
    Sigma,
    w_prev,
    gamma=5.0,
    lam_t=0.0020,
    eta=2.0,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3
):
    n = len(mu)
    w = cp.Variable(n)

    S = make_psd_matrix(Sigma, ridge=ridge)
    trade = w - w_prev

    objective = cp.Maximize(
        mu @ w
        - gamma * cp.quad_form(w, S)
        - lam_t * cp.norm1(trade)
        - eta * cp.sum_squares(trade)
    )

    constraints = [
        cp.sum(w) == 1,
        w >= 0,
        w <= w_max,
        cp.norm1(trade) <= tau,
    ]

    prob = cp.Problem(objective, constraints)
    ok = solve_problem(prob)

    if not ok:
        return normalise_weights(w_prev)

    return normalise_weights(w.value, fallback=w_prev)


# ============================================================
# 8) ENHANCED two-step
#    Step 1: target w*
#    Step 2: min (rho/2)||w-w*||^2 + lam_t ||w-w_prev||_1 + eta ||w-w_prev||_2^2
#            s.t. ||w-w_prev||_1 <= tau
# ============================================================
def solve_enhanced_two_step(
    mu,
    Sigma,
    w_prev,
    gamma=5.0,
    rho=25.0,
    lam_t=0.0020,
    eta=2.0,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3
):
    n = len(mu)
    S = make_psd_matrix(Sigma, ridge=ridge)

    # Step 1: target portfolio
    w_star = cp.Variable(n)
    objective1 = cp.Maximize(
        mu @ w_star - gamma * cp.quad_form(w_star, S)
    )
    constraints1 = add_constraints(
        w_star,
        long_only=True,
        w_max=w_max,
        fully_invested=True
    )

    prob1 = cp.Problem(objective1, constraints1)
    ok1 = solve_problem(prob1)

    if not ok1:
        w_star_val = normalise_weights(w_prev)
    else:
        w_star_val = normalise_weights(w_star.value, fallback=w_prev)

    # Step 2: enhanced trade-towards-target problem
    w = cp.Variable(n)
    trade = w - w_prev

    objective2 = cp.Minimize(
        (rho / 2.0) * cp.sum_squares(w - w_star_val)
        + lam_t * cp.norm1(trade)
        + eta * cp.sum_squares(trade)
    )

    constraints2 = [
        cp.sum(w) == 1,
        w >= 0,
        w <= w_max,
        cp.norm1(trade) <= tau,
    ]

    prob2 = cp.Problem(objective2, constraints2)
    ok2 = solve_problem(prob2)

    if not ok2:
        w_val = normalise_weights(w_prev)
    else:
        w_val = normalise_weights(w.value, fallback=w_prev)

    return w_star_val, w_val


# ============================================================
# 9) Run one of the four strategies through time
# ============================================================
def run_strategy_combo(
    mu_series,
    Sigma_series,
    cost_series,
    family="basic",      # "basic" or "enhanced"
    method="one",        # "one" or "two"
    gamma=5.0,
    rho=25.0,
    eta=2.0,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3,
    w0=None
):
    T, n = mu_series.shape

    if w0 is None:
        w_prev = np.ones(n) / n
    else:
        w_prev = normalise_weights(w0)

    W = np.zeros((T, n))
    W_star = np.full((T, n), np.nan)

    for t in range(T):
        mu_t = mu_series[t]
        Sigma_t = Sigma_series[t]
        lam_t = cost_series[t]

        if family == "basic" and method == "one":
            w_t = solve_basic_one_step(
                mu=mu_t,
                Sigma=Sigma_t,
                w_prev=w_prev,
                gamma=gamma,
                lam_t=lam_t,
                w_max=w_max,
                ridge=ridge
            )

        elif family == "basic" and method == "two":
            w_star_t, w_t = solve_basic_two_step(
                mu=mu_t,
                Sigma=Sigma_t,
                w_prev=w_prev,
                gamma=gamma,
                rho=rho,
                lam_t=lam_t,
                w_max=w_max,
                ridge=ridge
            )
            W_star[t] = w_star_t

        elif family == "enhanced" and method == "one":
            w_t = solve_enhanced_one_step(
                mu=mu_t,
                Sigma=Sigma_t,
                w_prev=w_prev,
                gamma=gamma,
                lam_t=lam_t,
                eta=eta,
                tau=tau,
                w_max=w_max,
                ridge=ridge
            )

        elif family == "enhanced" and method == "two":
            w_star_t, w_t = solve_enhanced_two_step(
                mu=mu_t,
                Sigma=Sigma_t,
                w_prev=w_prev,
                gamma=gamma,
                rho=rho,
                lam_t=lam_t,
                eta=eta,
                tau=tau,
                w_max=w_max,
                ridge=ridge
            )
            W_star[t] = w_star_t

        else:
            raise ValueError("Invalid combination of family and method.")

        W[t] = w_t
        w_prev = w_t

    return W, W_star


# ============================================================
# 10) Realised returns
# ============================================================
def realised_returns_gross(R, W):
    return np.sum(R[1:] * W[:-1], axis=1)


def realised_returns_net(R, W, cost_series):
    gross = realised_returns_gross(R, W)
    T = W.shape[0]

    trade_costs = np.zeros(T - 1)
    for k in range(1, T - 1):
        trn = np.sum(np.abs(W[k] - W[k - 1]))
        trade_costs[k] = cost_series[k] * trn

    net = gross - trade_costs
    return gross, net, trade_costs


# ============================================================
# 11) Summarise one strategy
# ============================================================
def summarise_strategy(name, W, W_star, R_daily, cost_series):
    to = turnover_from_weights(W)
    rp_gross, rp_net, tc_paid = realised_returns_net(R_daily, W, cost_series)

    eq_gross = equity_curve(rp_gross)
    eq_net = equity_curve(rp_net)

    stats_gross = summary_stats(rp_gross)
    stats_net = summary_stats(rp_net)

    out = {
        "name": name,
        "W": W,
        "W_star": W_star,
        "turnover": to,
        "rp_gross": rp_gross,
        "rp_net": rp_net,
        "tc_paid": tc_paid,
        "eq_gross": eq_gross,
        "eq_net": eq_net,
        "stats_gross": stats_gross,
        "stats_net": stats_net,
        "avg_turnover": np.mean(to) if len(to) > 0 else np.nan,
        "max_turnover": np.max(to) if len(to) > 0 else np.nan,
        "avg_max_weight": float(np.mean(np.max(W, axis=1))) if W.size > 0 else np.nan,
        "has_nan_weights": bool(np.isnan(W).any()),
        "avg_cost_coeff": float(np.mean(cost_series)),
        "avg_realised_tc": float(np.mean(tc_paid)) if len(tc_paid) > 0 else np.nan,
    }
    return out


# ============================================================
# 12) Plot helpers
# ============================================================
def plot_all_results(times, results, cost_series):
    date_index_full = pd.to_datetime(times)
    date_index = pd.to_datetime(times[1:])

    plt.figure(figsize=(10, 5))
    plt.plot(date_index_full, cost_series)
    plt.title("Time-varying transaction cost coefficient")
    plt.xlabel("Date")
    plt.ylabel("Cost coefficient")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    for r in results:
        plt.plot(date_index, r["eq_net"], label=r["name"])
    plt.title("Net equity curves")
    plt.xlabel("Date")
    plt.ylabel("Equity")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    for r in results:
        plt.plot(date_index, r["eq_gross"], label=r["name"])
    plt.title("Gross equity curves")
    plt.xlabel("Date")
    plt.ylabel("Equity")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    for r in results:
        plt.plot(date_index, r["turnover"], label=r["name"])
    plt.title("Turnover through time")
    plt.xlabel("Date")
    plt.ylabel("Turnover")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    for r in results:
        plt.plot(date_index, r["tc_paid"], label=r["name"])
    plt.title("Realised trading cost through time")
    plt.xlabel("Date")
    plt.ylabel("Trading cost")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Weight heatmaps
    for r in results:
        plt.figure(figsize=(10, 4))
        plt.imshow(r["W"].T, aspect="auto")
        plt.title(f"Weights heatmap | {r['name']}")
        plt.xlabel("Time")
        plt.ylabel("Asset")
        plt.colorbar(label="Weight")
        plt.tight_layout()
        plt.show()

        if not np.isnan(r["W_star"]).all():
            plt.figure(figsize=(10, 4))
            plt.imshow(r["W_star"].T, aspect="auto")
            plt.title(f"Target weights heatmap | {r['name']}")
            plt.xlabel("Time")
            plt.ylabel("Asset")
            plt.colorbar(label="Target weight")
            plt.tight_layout()
            plt.show()


# ============================================================
# 13) Run one seed for all 4 combinations
# ============================================================
def run_single_seed_four_combinations(
    seed=7,
    n_assets=50,
    lookback=252,
    use_log_returns=False,
    gamma=5.0,
    rho=25.0,
    eta=2.0,
    tau=0.05,
    w_max=0.10,
    ridge=1e-3,
    base_cost=0.0020,
    cost_mode="vol_scaled",
    vol_lookback=21,
    alpha=2.0,
    annualise=252
):
    data = qndata.stocks.load_data()

    data, chosen_assets = load_quantiacs_universe(
        seed=seed,
        n_assets=n_assets,
        data=data
    )

    print("Chosen assets (first 10):", chosen_assets[:10], "... total:", len(chosen_assets))

    times, mu_series, Sigma_series, R_daily = compute_mu_sigma_series(
        data=data,
        chosen_assets=chosen_assets,
        lookback=lookback,
        annualise=annualise,
        use_log_returns=use_log_returns
    )

    cost_series = build_time_varying_cost_series(
        R_daily,
        base_cost=base_cost,
        mode=cost_mode,
        vol_lookback=vol_lookback,
        alpha=alpha
    )

    combos = [
        ("Basic + One-Step", "basic", "one"),
        ("Basic + Two-Step", "basic", "two"),
        ("Enhanced + One-Step", "enhanced", "one"),
        ("Enhanced + Two-Step", "enhanced", "two"),
    ]

    results = []

    for name, family, method in combos:
        print(f"Running {name} ...")

        W, W_star = run_strategy_combo(
            mu_series=mu_series,
            Sigma_series=Sigma_series,
            cost_series=cost_series,
            family=family,
            method=method,
            gamma=gamma,
            rho=rho,
            eta=eta,
            tau=tau,
            w_max=w_max,
            ridge=ridge
        )

        res = summarise_strategy(
            name=name,
            W=W,
            W_star=W_star,
            R_daily=R_daily,
            cost_series=cost_series
        )
        results.append(res)

    rows = []
    for r in results:
        rows.append({
            "strategy": r["name"],
            "final_equity_gross": r["eq_gross"][-1] if len(r["eq_gross"]) > 0 else np.nan,
            "final_equity_net": r["eq_net"][-1] if len(r["eq_net"]) > 0 else np.nan,
            "cum_return_gross": r["stats_gross"]["cum_return"],
            "cum_return_net": r["stats_net"]["cum_return"],
            "sharpe_gross": r["stats_gross"]["sharpe"],
            "sharpe_net": r["stats_net"]["sharpe"],
            "max_drawdown_gross": r["stats_gross"]["max_drawdown"],
            "max_drawdown_net": r["stats_net"]["max_drawdown"],
            "avg_turnover": r["avg_turnover"],
            "max_turnover": r["max_turnover"],
            "avg_max_weight": r["avg_max_weight"],
            "avg_cost_coeff": r["avg_cost_coeff"],
            "avg_realised_tc": r["avg_realised_tc"],
            "has_nan_weights": r["has_nan_weights"],
        })

    summary_df = pd.DataFrame(rows)

    print("\n=== Summary table ===")
    print(summary_df)

    for r in results:
        print_stats(f"{r['name']} | Gross stats", r["stats_gross"])
        print_stats(f"{r['name']} | Net stats", r["stats_net"])

    plot_all_results(times, results, cost_series)

    return {
        "seed": seed,
        "chosen_assets": chosen_assets,
        "times": times,
        "mu_series": mu_series,
        "Sigma_series": Sigma_series,
        "R_daily": R_daily,
        "cost_series": cost_series,
        "results": results,
        "summary_df": summary_df,
    }


# ============================================================
# 14) Main
# ============================================================
if __name__ == "__main__":
    out = run_single_seed_four_combinations(
        seed=7,
        n_assets=50,
        lookback=252,
        use_log_returns=False,
        gamma=5.0,
        rho=25.0,
        eta=2.0,
        tau=0.05,
        w_max=0.10,
        ridge=1e-3,
        base_cost=0.0020,
        cost_mode="vol_scaled",   # or "constant"
        vol_lookback=21,
        alpha=2.0,
        annualise=252
    )